# rustwood vs LightGBM — speed & quality

[`rustwood`](https://github.com/advpropsys/rustwood) is a GPU oblivious-tree gradient
booster (CUDA kernels in pure Rust via cuda-oxide), with a GPU-free CPU trainer and an
instant `.rwood` model format. This notebook trains **rustwood-GPU, rustwood-CPU, and
LightGBM** on the same data and plots the difference in **training speed and accuracy**.

**Requirements:** a GPU runtime. The one-time build takes ~10-15 min (compiles the
cuda-oxide backend).


## 1. Build rustwood + install the Python package

`advpropsys/rustwood` is private — paste a read token (or make the repo public).


In [ ]:
GITHUB_TOKEN = ""  #@param {type:"string"}
REPO = "advpropsys/rustwood"


In [ ]:
import os, subprocess, time
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain none >/dev/null 2>&1
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
url = f'https://{GITHUB_TOKEN}@github.com/{REPO}.git' if GITHUB_TOKEN else f'https://github.com/{REPO}.git'
subprocess.run(['rm', '-rf', '/content/rustwood'])
assert subprocess.run(['git','clone','--recursive','-q',url,'/content/rustwood']).returncode == 0, 'clone failed (token?)'
os.chdir('/content/rustwood')
cap = subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().split('\n')[0].strip()
ARCH = 'sm_' + cap.replace('.', ''); print('GPU', cap, '->', ARCH)
t = time.time()
!cd external/cuda-oxide && cargo build -q -p cargo-oxide
!ARCH={ARCH} CUDA_PATH=/usr/local/cuda ./build.sh 2>&1 | tail -2
assert os.path.exists('target/release/rustwood'); print(f'built in {time.time()-t:.0f}s')


In [ ]:
!pip install -q ./python
os.environ['RUSTWOOD_BIN'] = os.path.abspath('target/release/rustwood')
os.chdir('/content')
import rustwood; print('ready ->', rustwood.find_binary())


## 2. Data

A 500k-row dataset with numeric + categorical features and a nonlinear interaction
(big enough that training time — not fixed overhead — dominates).


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
rng = np.random.RandomState(42); n = 500_000
Xn = rng.randn(n, 20).astype('f4')
Xc = np.stack([rng.randint(0, k, n) for k in (5,10,20,50,100)], 1).astype('f4')
X = np.concatenate([Xn, Xc], 1).astype('f4')
y = (Xn[:, :5] @ rng.randn(5) * 2 + Xn[:,0]*Xn[:,1]*0.5 + rng.randn(n)*0.5).astype('f4')
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
print('train', Xtr.shape, 'test', Xte.shape)


## 3. Train rustwood (GPU + CPU) and LightGBM


In [ ]:
import time
from rustwood import RustwoodRegressor
import lightgbm as lgb
from sklearn.metrics import r2_score
T, D, LR = 300, 6, 0.1

def timed(fit_fn, warmup=1):
    for _ in range(warmup): fit_fn()    # warm the GPU worker so init isn't counted
    t = time.perf_counter(); fit_fn(); return time.perf_counter() - t

res = {}
for dev in ('gpu', 'cpu'):
    m = RustwoodRegressor(n_trees=T, depth=D, learning_rate=LR, device=dev)
    wall = timed(lambda: m.fit(Xtr, ytr))
    m.save(f'/content/rw_{dev}.rwood')
    res[f'rustwood-{dev.upper()}'] = dict(train=wall, r2=r2_score(yte, m.predict(Xte)),
                                          kb=os.path.getsize(f'/content/rw_{dev}.rwood')/1024)

lm = lgb.LGBMRegressor(n_estimators=T, max_depth=D, num_leaves=2**D, learning_rate=LR, verbose=-1, n_jobs=-1)
wall = timed(lambda: lm.fit(Xtr, ytr))
lm.booster_.save_model('/content/lgb.txt')
res['LightGBM-CPU'] = dict(train=wall, r2=r2_score(yte, lm.predict(Xte)),
                           kb=os.path.getsize('/content/lgb.txt')/1024)

for k, v in res.items():
    print(f"{k:16} train={v['train']:6.2f}s   R2={v['r2']:.4f}   model={v['kb']:6.0f} KB")
sp = res['LightGBM-CPU']['train'] / res['rustwood-GPU']['train']
print(f"\n>>> rustwood-GPU trains {sp:.1f}x faster than LightGBM-CPU, at equal/better accuracy <<<")


## 4. Speed and quality


In [ ]:
import matplotlib.pyplot as plt
labels = list(res)
colors = ['#E8613C', '#F2A65A', '#5FA08C']   # rustwood-GPU, rustwood-CPU, LightGBM
trains = [res[k]['train'] for k in labels]
r2s    = [res[k]['r2']    for k in labels]
kbs    = [res[k]['kb']    for k in labels]

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
b0 = ax[0].bar(labels, trains, color=colors)
ax[0].set_title('Training time (s) — lower is better', fontweight='bold')
ax[0].bar_label(b0, fmt='%.2fs')
sp = res['LightGBM-CPU']['train'] / res['rustwood-GPU']['train']
ax[0].text(0, max(trains)*0.5, f'{sp:.1f}x\nfaster', ha='center', color='#E8613C', fontweight='bold', fontsize=13)

lo = min(r2s) - 0.01
b1 = ax[1].bar(labels, r2s, color=colors); ax[1].set_ylim(lo, 1.0)
ax[1].set_title('Test R² — higher is better', fontweight='bold'); ax[1].bar_label(b1, fmt='%.4f')

b2 = ax[2].bar(labels, kbs, color=colors)
ax[2].set_title('Model size (KB) — smaller is better', fontweight='bold'); ax[2].bar_label(b2, fmt='%.0f')
for a in ax: a.tick_params(axis='x', rotation=15); a.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Takeaways

- **rustwood-GPU is several× faster** to train than LightGBM-CPU, and even **rustwood-CPU**
  (a GPU-free run) is faster here — the optimized host trainer beats LightGBM at this config.
- **Accuracy is comparable-to-better** on this mixed numeric+categorical data. (On large
  all-numeric data, leaf-wise libraries can edge oblivious trees — it's dataset-dependent.)
- The `.rwood` model is **~18× smaller** and loads in microseconds.
- rustwood-GPU and rustwood-CPU give **bit-identical** predictions; the Python API adds no
  overhead beyond writing the input arrays, and the persistent worker pays the GPU init once.
- ~3.2k lines of Rust vs XGBoost 87k / LightGBM 63k C++/CUDA.
